In [ ]:
import pandas as pd
import glob
import numpy as np
from datetime import datetime, timedelta

# --- 1. LOAD & CLEAN NASA DATA ---
# Your NASA file has a 15-line header. 
nasa_df = pd.read_csv('../data/raw/NASA Power DAV.csv', skiprows=15)

# Convert YEAR and DOY (Day of Year) into a single 'date' column
nasa_df['date'] = nasa_df.apply(lambda row: datetime(int(row['YEAR']), 1, 1) + timedelta(days=int(row['DOY']) - 1), axis=1)
nasa_df.replace(-999, np.nan, inplace=True) # Replace NASA missing values with NaN

# --- 2. LOAD & CLEAN NOAA DATA ---
noaa_df = pd.read_csv('../data/raw/NOAA.csv')
noaa_df['date'] = pd.to_datetime(noaa_df['DATE'])

# --- 3. LOAD & CONCATENATE TRADE DATA ---
# This looks for all files starting with "TradeData" and joins them
trade_files = glob.glob('../data/raw/TradeData*.csv')
trade_list = [pd.read_csv(f) for f in trade_files]
trade_df = pd.concat(trade_list, ignore_index=True)

# Trade data is yearly; we'll create a 'year' column to merge later
trade_df['year'] = trade_df['refYear']

# --- 4. LOAD WFP DATA (The 215MB Zip) ---
wfp_df = pd.read_csv('../data/raw/globalfoodprices_wfp.zip') 
wfp_df['date'] = pd.to_datetime(wfp_df['date'])
wfp_df['year'] = wfp_df['date'].dt.year

# --- 5. THE MASTER MERGE ---
# First, merge the two weather sources (NASA + NOAA)
weather_combined = pd.merge(nasa_df, noaa_df, on='date', how='outer')

# Merge Weather with Price Data (WFP)
# We merge on 'date' to align daily weather with price points
master_df = pd.merge(wfp_df, weather_combined, on='date', how='left')

# Finally, merge with Trade context (Yearly)
final_df = pd.merge(master_df, trade_df, on=['year'], how='left', suffixes=('', '_trade'))

# --- 6. VALIDATION ---
print(f"Total Rows: {final_df.shape[0]}")
if final_df.shape[0] >= 500000:
    print("✅ 500,000 Row Requirement Met!")

# Export the final result
# final_df.to_csv('../data/processed/final_master_dataset.csv', index=False)